In [ ]:
# ============================================================
# CONFIGURAÇÃO DO SPARK COM CREDENCIAIS AWS
# ============================================================
#
# Este notebook testa a conexão entre o ambiente local, o Spark
# e o bucket S3 configurado para o integrante que está executando
# o projeto.
#
# Cada integrante deve usar suas próprias credenciais do AWS Academy
# Learner Lab e seu próprio bucket S3.
#
# As credenciais e o bucket não devem ser escritos diretamente no
# código-fonte. Eles devem ser carregados a partir de variáveis de
# ambiente, geralmente definidas no arquivo .env.
#
# Pontos de configuração esperados:
#
#   AWS_ACCESS_KEY_ID      Chave de acesso temporária da AWS.
#   AWS_SECRET_ACCESS_KEY  Chave secreta temporária da AWS.
#   AWS_SESSION_TOKEN      Token temporário da sessão AWS Academy.
#   S3_BUCKET              Nome do bucket S3 do integrante.
#
# Saída:
#
#   spark    Sessão Spark configurada para acessar o S3.
#   S3_PATH  Caminho usado no teste de escrita e leitura no bucket.
#
# ============================================================
# BIBLIOTECAS
# ============================================================

from pyspark.sql import SparkSession
import os

# ============================================================
# VARIÁVEIS DE AMBIENTE
# ============================================================

# Carrega as credenciais temporárias e o bucket a partir do ambiente.
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
S3_BUCKET = os.getenv("S3_BUCKET")

# Caminho no S3 usado exclusivamente para validar a conexão.
S3_PATH = f"s3a://{S3_BUCKET}/_teste_conexao/teste"

# ============================================================
# SESSÃO SPARK
# ============================================================

# Cria uma sessão Spark local com suporte ao conector S3A.
# O TemporaryAWSCredentialsProvider é necessário porque o AWS Academy
# usa credenciais temporárias com session token.
spark = (
    SparkSession.builder
    .appName("pipeline-alfabetizacao")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider"
    )
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN)
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com")
    .getOrCreate()
)

In [ ]:
# ============================================================
# TESTE DE ESCRITA E LEITURA NO S3
# ============================================================
#
# Esta célula valida se a sessão Spark consegue escrever e ler dados
# no bucket S3 configurado para o integrante.
#
# O teste cria um pequeno DataFrame em memória, grava esse dado em
# formato Parquet no caminho de teste e, em seguida, lê o mesmo caminho
# para confirmar que a conexão está funcionando.
#
# Entrada:
#
#   spark    Sessão Spark configurada com credenciais AWS.
#   S3_PATH  Caminho de teste no bucket S3 do integrante.
#
# Saída:
#
#   Um pequeno DataFrame lido de volta do S3.
#
# ============================================================
# CRIAÇÃO DOS DADOS DE TESTE
# ============================================================

# Cria um DataFrame simples apenas para validar a escrita e leitura.
data = [("Teste de Conexao", 1)]
df = spark.createDataFrame(data, ["parametro", "valor"])

# ============================================================
# ESCRITA NO S3
# ============================================================

# Grava o DataFrame no S3 em formato Parquet.
# O modo overwrite evita erro caso o teste já tenha sido executado antes.
df.write.mode("overwrite").parquet(S3_PATH)

# ============================================================
# LEITURA E VERIFICAÇÃO
# ============================================================

# Lê o arquivo gravado no S3 para confirmar que a escrita funcionou.
df = spark.read.parquet(S3_PATH)

# Exibe o resultado do teste.
df.show()